# **Main Finetune**

In [ ]:
# ============================================================
# PLANNER FINETUNING — Mistral-7B-Instruct-v0.3
# Task   : ANSWER / ASK / ABSTAIN decision planner
# GPU    : L4 (24GB VRAM) | bf16 | No quantization
# LoRA   : r=32, alpha=64
# ============================================================

import os, gc, json, torch, warnings, subprocess, sys
warnings.filterwarnings("ignore")

print("=" * 60)
print("  PLANNER FINETUNING — Mistral-7B-Instruct-v0.3")
print("=" * 60)


# ── 1. Install Dependencies ──────────────────────────────────

def silent_install(pkg):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg]
    )

required = {
    "transformers": "transformers>=4.40.0",
    "peft"        : "peft>=0.10.0",
    "trl"         : "trl>=0.8.6",
    "accelerate"  : "accelerate>=0.29.0",
    "datasets"    : "datasets>=2.18.0",
}

print("\n  Checking libraries...")
for lib, pkg in required.items():
    try:
        __import__(lib)
        print(f"  ✅  {lib:<20} ready")
    except ImportError:
        print(f"  ⬇️   {lib:<20} installing...")
        silent_install(pkg)
        print(f"  ✅  {lib:<20} installed")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import trl
print(f"\n  TRL version: {trl.__version__}")


# ── 2. Paths ─────────────────────────────────────────────────

FT_OUT_DIR   = "/content/ft_dataset"
GDRIVE_PATH  = "/content/drive/MyDrive/planner_finetune"
OUTPUT_DIR   = "/content/mistral_planner_output"
ADAPTER_DIR  = "/content/mistral_planner_adapter"

TRAIN_FILE = "/content/ft_train_mistral.jsonl"
VAL_FILE   = "/content/ft_val_mistral.jsonl"

for d in [OUTPUT_DIR, ADAPTER_DIR]:
    os.makedirs(d, exist_ok=True)

# mount drive early so saves go there immediately
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
os.makedirs(GDRIVE_PATH, exist_ok=True)
print(f"  Drive mounted → {GDRIVE_PATH}")


# ── 3. GPU Check ─────────────────────────────────────────────

print("\n" + "=" * 60)
print("  GPU CHECK")
print("=" * 60)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Switch runtime.")

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"  GPU     : {gpu_name}")
print(f"  VRAM    : {gpu_mem_gb:.1f} GB")
print(f"  CUDA    : {torch.version.cuda}")
print(f"  PyTorch : {torch.__version__}fo")

USE_BF16    = torch.cuda.is_bf16_supported()
USE_FP16    = not USE_BF16
BATCH_SIZE  = 8      # L4 24GB, no quantization
GRAD_ACCUM  = 2      # effective batch = 16
MAX_SEQ_LEN = 512    # covers multi-turn samples safely

print(f"\n  Precision      : {'bf16' if USE_BF16 else 'fp16'}")
print(f"  batch_size     : {BATCH_SIZE}")
print(f"  grad_accum     : {GRAD_ACCUM} → eff batch = {BATCH_SIZE*GRAD_ACCUM}")
print(f"  max_seq_len    : {MAX_SEQ_LEN}")


In [ ]:

# ── Dialogue-preserving subsample ────────────────────────────
def subsample_preserving_dialogues(path, target_total):
    import json, random
    from collections import defaultdict
    random.seed(42)
    by_dialogue, single_turn = defaultdict(list), []
    with open(path) as f:
        for line in f:
            s = json.loads(line)
            dlg = s.get("dialogue_id")
            (by_dialogue[dlg] if dlg else single_turn).append(s)

    def get_action(s):
        c = s["messages"][2]["content"]
        if "<decision>\nANSWER" in c: return "ANSWER"
        if "<decision>\nASK"    in c: return "ASK"
        return "ABSTAIN"

    dlg_by_action = defaultdict(list)
    for dlg_id, turns in by_dialogue.items():
        acts = [get_action(t) for t in turns]
        dlg_by_action[max(set(acts), key=acts.count)].append(dlg_id)

    single_by_action = defaultdict(list)
    for s in single_turn:
        single_by_action[get_action(s)].append(s)

    action_targets = {
        "ANSWER" : int(target_total * 0.30),
        "ASK"    : int(target_total * 0.38),
        "ABSTAIN": int(target_total * 0.32),
    }
    selected = []
    for action, target in action_targets.items():
        count = 0
        dlg_ids = dlg_by_action.get(action, [])
        random.shuffle(dlg_ids)
        for dlg_id in dlg_ids:
            turns = by_dialogue[dlg_id]
            if count + len(turns) <= target:
                selected.extend(turns); count += len(turns)
            if count >= target: break
        singles = single_by_action.get(action, [])
        random.shuffle(singles)
        rem = target - count
        if rem > 0: selected.extend(singles[:rem])
        print(f"  {action:10}: {min(count+rem, target)} samples")

    random.shuffle(selected)
    tmp = path.replace(".jsonl", "_sub.jsonl")
    with open(tmp, "w") as f:
        for s in selected: f.write(json.dumps(s) + "\n")
    print(f"  Total: {len(selected)} → {tmp}")
    return tmp

print("\nSubsampling datasets (dialogue-preserving)...")
TRAIN_FILE = subsample_preserving_dialogues(TRAIN_FILE, target_total=9000)
VAL_FILE   = subsample_preserving_dialogues(VAL_FILE,   target_total=1200)

# ── 4. Load Model + Tokenizer ────────────────────────────────

print("\n" + "=" * 60)
print("  LOADING MODEL + TOKENIZER")
print("=" * 60)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

try:
    model
    tokenizer
    print("  ✅  Already in memory — skipping reload")
except NameError:
    print(f"  Loading: {MODEL_ID}")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID, trust_remote_code=True
    )
    print("  ✅  Tokenizer loaded")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype       = torch.bfloat16 if USE_BF16 else torch.float16,
        device_map        = "auto",
        trust_remote_code = True,
    )
    print("  ✅  Model loaded")

tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

total_params = sum(p.numel() for p in model.parameters())
print(f"\n  pad_token    : {tokenizer.pad_token}")
print(f"  chat_template: {'present ✅' if tokenizer.chat_template else 'MISSING ⚠️'}")
print(f"  Total params : {total_params/1e9:.2f}B")


# ── 5. Load + Format Dataset ─────────────────────────────────

print("\n" + "=" * 60)
print("  LOADING DATASETS")
print("=" * 60)

# ft_train_mistral.jsonl and ft_val_mistral.jsonl already have
# {"messages": [...]} format from dataset builder
train_ds_raw = load_dataset(
    "json", data_files=TRAIN_FILE, split="train"
)
val_ds_raw = load_dataset(
    "json", data_files=VAL_FILE, split="train"
)

print(f"  Train : {len(train_ds_raw)} samples")
print(f"  Val   : {len(val_ds_raw)} samples")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize              = False,
        add_generation_prompt = False,
    )
    tokens = tokenizer(
        text, truncation=True, max_length=512, return_tensors=None
    )
    return {"text": tokenizer.decode(
        tokens["input_ids"], skip_special_tokens=False
    )}

print("\n  Applying chat template...")
train_ds = train_ds_raw.map(format_chat, batched=False)
val_ds   = val_ds_raw.map(format_chat, batched=False)
print("  ✅  Done")

# token length check
sample_lens = [
    len(tokenizer(s["text"])["input_ids"])
    for s in train_ds.select(range(min(500, len(train_ds))))
]
over = sum(1 for l in sample_lens if l > MAX_SEQ_LEN)
print(f"\n  Token lengths (500-sample check):")
print(f"    min={min(sample_lens)} "
      f"avg={sum(sample_lens)//len(sample_lens)} "
      f"max={max(sample_lens)}")
print(f"    > {MAX_SEQ_LEN} tokens : {over} "
      f"({'will truncate' if over else '✅ none'})")

print("\n  Sample (first 400 chars):")
print("  " + "─"*50)
print(train_ds[0]["text"][:400])
print("  " + "─"*50)


# ── 6. LoRA Configuration ────────────────────────────────────

print("\n" + "=" * 60)
print("  LoRA CONFIGURATION")
print("=" * 60)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print("  ✅  Gradient checkpointing enabled")

lora_config = LoraConfig(
    r              = 32,
    lora_alpha     = 64,
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
    target_modules = [
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj",
        "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
pct        = 100 * trainable / all_params

print(f"\n  r            : {lora_config.r}")
print(f"  alpha        : {lora_config.lora_alpha}")
print(f"  dropout      : {lora_config.lora_dropout}")
print(f"  Trainable    : {trainable:,}  ({pct:.3f}%)")
print(f"  Frozen       : {all_params - trainable:,}")


# ── 7. Training Arguments ────────────────────────────────────

print("\n" + "=" * 60)
print("  TRAINING ARGUMENTS")
print("=" * 60)

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 2,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    bf16                        = USE_BF16,
    fp16                        = USE_FP16,
    logging_dir                 = f"{OUTPUT_DIR}/logs",
    logging_steps               = 20,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_pin_memory       = False,
    optim                       = "adamw_torch",
    max_grad_norm               = 1.0,
    weight_decay                = 0.01,
    group_by_length             = False,
    dataset_text_field          = "text",
    # max_seq_length              = MAX_SEQ_LEN,
)

steps_per_epoch = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * int(training_args.num_train_epochs)

for k, v in {
    "epochs"       : training_args.num_train_epochs,
    "train_batch"  : training_args.per_device_train_batch_size,
    "grad_accum"   : training_args.gradient_accumulation_steps,
    "eff_batch"    : BATCH_SIZE * GRAD_ACCUM,
    "learning_rate": training_args.learning_rate,
    "lr_scheduler" : training_args.lr_scheduler_type,
    "warmup_ratio" : training_args.warmup_ratio,
    "steps/epoch"  : steps_per_epoch,
    "total_steps"  : total_steps,
    "eval_strategy": training_args.eval_strategy,
    "output_dir"   : training_args.output_dir,
}.items():
    print(f"  {k:<22}: {v}")


# ── 8. Callbacks ─────────────────────────────────────────────

class PlannerCallback(TrainerCallback):
    """
    Logs training progress and saves adapter to Drive
    after every epoch so nothing is lost mid-training.
    """
    def __init__(self):
        self.train_losses = []
        self.eval_losses  = []
        self.best_eval    = float("inf")
        self.best_epoch   = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step      = state.global_step
        loss      = logs.get("loss")
        eval_loss = logs.get("eval_loss")
        lr        = logs.get("learning_rate", 0)

        if loss is not None:
            self.train_losses.append((step, loss))
            if step % 100 == 0 or step <= 20:
                print(f"  step {step:>5} | "
                      f"train_loss: {loss:.4f} | "
                      f"lr: {lr:.2e}")

        if eval_loss is not None:
            self.eval_losses.append((step, eval_loss))
            marker = ""
            if eval_loss < self.best_eval:
                self.best_eval  = eval_loss
                self.best_epoch = state.epoch
                marker          = "  ← best ✅"
            print(f"\n  {'─'*54}")
            print(f"  EPOCH {state.epoch:.0f} | "
                  f"eval_loss: {eval_loss:.4f}{marker}")
            print(f"  {'─'*54}\n")

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch = int(state.epoch) + 1
        print(f"\n  ════ EPOCH {epoch} / "
              f"{int(args.num_train_epochs)} ════\n")

    def on_epoch_end(self, args, state, control, **kwargs):
        """Save adapter to Drive after every epoch."""
        import shutil
        epoch = int(state.epoch)
        epoch_dir = f"{GDRIVE_PATH}/epoch_{epoch}"
        os.makedirs(epoch_dir, exist_ok=True)
        try:
            # save current adapter state to Drive
            model.save_pretrained(epoch_dir)
            tokenizer.save_pretrained(epoch_dir)
            print(f"\n  💾  Epoch {epoch} adapter saved → {epoch_dir}")
        except Exception as e:
            print(f"\n  ⚠️  Drive save failed: {e}")

    def on_train_end(self, args, state, control, **kwargs):
        print("\n" + "=" * 60)
        print("  TRAINING COMPLETE")
        print("=" * 60)
        if self.train_losses:
            print(f"  Final train loss : {self.train_losses[-1][1]:.4f}")
        print(f"  Best eval loss   : {self.best_eval:.4f} "
              f"(epoch {self.best_epoch:.0f})")
        print(f"  Total steps      : {state.global_step}")


planner_callback = PlannerCallback()


# ── 9. Trainer Setup ─────────────────────────────────────────

print("\n" + "=" * 60)
print("  INITIALIZING SFTTrainer")
print("=" * 60)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_ds,
    eval_dataset     = val_ds,
    args             = training_args,
    callbacks        = [planner_callback],
)
print(f"  ✅  SFTTrainer ready")
print(f"  Steps/epoch : {steps_per_epoch}")
print(f"  Total steps : {total_steps}")


# ── 10. Memory Check ─────────────────────────────────────────

gc.collect()
torch.cuda.empty_cache()

alloc  = torch.cuda.memory_allocated(0) / 1e9
reserv = torch.cuda.memory_reserved(0) / 1e9
free   = gpu_mem_gb - reserv

print(f"\n  GPU memory: allocated={alloc:.2f}GB | "
      f"reserved={reserv:.2f}GB | free={free:.2f}GB")

if free < 2.0:
    print("  ⚠️  Low VRAM — reduce BATCH_SIZE to 2 and rerun")
else:
    print("  ✅  Memory healthy — safe to train")


# ── 11. TRAIN ────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  STARTING TRAINING")
print("=" * 60 + "\n")

train_result = trainer.train()


# ── 12. Training Summary ─────────────────────────────────────

print("\n" + "=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)

m = train_result.metrics
print(f"  Runtime        : {m.get('train_runtime',0):.1f}s  "
      f"({m.get('train_runtime',0)/60:.1f} min)")
print(f"  Samples/sec    : {m.get('train_samples_per_second',0):.2f}")
print(f"  Final loss     : {m.get('train_loss',0):.4f}")

if planner_callback.eval_losses:
    print(f"\n  Eval loss per epoch:")
    for step, loss in planner_callback.eval_losses:
        marker = " ← best" if loss == planner_callback.best_eval else ""
        print(f"    step {step:>5} → {loss:.4f}{marker}")


# ── 13. Save Final Adapter ───────────────────────────────────

print("\n" + "=" * 60)
print("  SAVING FINAL ADAPTER")
print("=" * 60)

# local save
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6
print(f"  ✅  Local: {ADAPTER_DIR}/  ({size_mb:.1f} MB)")
for f in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    print(f"    {f:<45} {sz:.2f} MB")

# Drive save — final best model
import shutil
final_drive_dir = f"{GDRIVE_PATH}/final_adapter"
os.makedirs(final_drive_dir, exist_ok=True)
shutil.copytree(ADAPTER_DIR, final_drive_dir, dirs_exist_ok=True)
print(f"  ✅  Drive: {final_drive_dir}/")

# also save training config for reference
config_record = {
    "model_id"    : MODEL_ID,
    "lora_r"      : lora_config.r,
    "lora_alpha"  : lora_config.lora_alpha,
    "epochs"      : int(training_args.num_train_epochs),
    "batch_size"  : BATCH_SIZE,
    "grad_accum"  : GRAD_ACCUM,
    "lr"          : training_args.learning_rate,
    "max_seq_len" : MAX_SEQ_LEN,
    "train_samples": len(train_ds),
    "val_samples"  : len(val_ds),
    "best_eval_loss": planner_callback.best_eval,
    "best_epoch"   : planner_callback.best_epoch,
    "final_train_loss": m.get("train_loss", 0),
    "train_runtime_min": m.get("train_runtime", 0) / 60,
}
with open(f"{final_drive_dir}/training_config.json", "w") as f:
    json.dump(config_record, f, indent=2)
print(f"  ✅  Config saved → {final_drive_dir}/training_config.json")


# ── 14. Inference Test ───────────────────────────────────────

print("\n" + "=" * 60)
print("  INFERENCE TEST — PLANNER BEHAVIOUR")
print("=" * 60)

SYSTEM_PROMPT = """You are a decision planner for a question-answering system.

Your task: given a user query, search the knowledge graph for relevant nodes, evaluate what information is present and what is missing, then decide the correct action.

Decision logic:
- Search the graph for nodes matching the query subject and known variables
- If the graph contains a complete path connecting known entities to an answer → ANSWER
- If the graph contains the topic but key linking variables are missing → ASK (specify what is missing)
- If the graph has no relevant nodes or the topic is entirely absent → ABSTAIN

Output format:
<reasoning>
Step 1 — Query subject: ...
Step 2 — Graph search: ...
Step 3 — Variable check: ...
Step 4 — Decision rationale: ...
</reasoning>
<decision>ANSWER | ASK | ABSTAIN</decision>
<justification>One sentence grounded in graph evidence.</justification>"""

test_cases = [
    {
        "label"   : "ANSWER — entity present in graph",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": (
                "<query>\nWho wrote Fear and Loathing in Las Vegas?\n</query>\n\n"
                "<known_variables>\nFear and Loathing in Las Vegas\n</known_variables>\n\n"
                "<graph_context>\nthompson | write | fear and loathing in las vegas\n"
                "thompson | publish_in | rolling stone\n</graph_context>\n\n"
                "<missing_variables>\nnone\n</missing_variables>\n\n"
                "Search the graph context and decide the correct action."
            )},
        ],
        "expect": "ANSWER",
    },
    {
        "label"   : "ASK — entity present, key detail missing",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": (
                "<query>\nWhen did Thompson write the book?\n</query>\n\n"
                "<known_variables>\nThompson, book\n</known_variables>\n\n"
                "<graph_context>\nthompson | write | book\n"
                "thompson | requires | ?unknown_1\n</graph_context>\n\n"
                "<missing_variables>\n- specific year of writing\n</missing_variables>\n\n"
                "Search the graph context and decide the correct action."
            )},
        ],
        "expect": "ASK",
    },
    {
        "label"   : "ABSTAIN — graph has no relevant nodes",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": (
                "<query>\nWhat is the GDP of Iceland?\n</query>\n\n"
                "<known_variables>\nIceland, GDP\n</known_variables>\n\n"
                "<graph_context>\nNo relevant nodes found in knowledge graph.\n</graph_context>\n\n"
                "<missing_variables>\nnone\n</missing_variables>\n\n"
                "Search the graph context and decide the correct action."
            )},
        ],
        "expect": "ABSTAIN",
    },
    {
        "label"   : "ASK — multi-turn, variable accumulation",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": (
                "<conversation_history>\n"
                "Turn 1 | ASK | Q: \"When was the album released?\" | "
                "A: [Clarification requested: 'release date' is needed]\n"
                "  → resolved: 'release date'\n"
                "</conversation_history>\n\n"
                "<resolved_variables>\n- release date\n</resolved_variables>\n\n"
                "<query>\nHow did the album sell?\n</query>\n\n"
                "<known_variables>\nalbum, release date\n</known_variables>\n\n"
                "<graph_context>\nbest album of | requires | ?unknown_1\n"
                "albums for the | requires | ?unknown_2\n</graph_context>\n\n"
                "<missing_variables>\n- sales performance\n</missing_variables>\n\n"
                "Search the graph context and decide the correct action."
            )},
        ],
        "expect": "ASK",
    },
    {
        "label"   : "ABSTAIN — multi-turn, vague query no graph",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": (
                "<query>\nIs there anything else interesting?\n</query>\n\n"
                "<known_variables>\nnone identified\n</known_variables>\n\n"
                "<graph_context>\nNo relevant nodes found in knowledge graph.\n</graph_context>\n\n"
                "<missing_variables>\nnone\n</missing_variables>\n\n"
                "Search the graph context and decide the correct action."
            )},
        ],
        "expect": "ABSTAIN",
    },
]

model.eval()
correct = 0

for i, case in enumerate(test_cases):
    input_text = tokenizer.apply_chat_template(
        case["messages"],
        tokenize              = False,
        add_generation_prompt = True,
    )
    inputs = tokenizer(
        input_text,
        return_tensors = "pt",
        truncation     = True,
        max_length     = MAX_SEQ_LEN,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 200,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

    gen  = out[0][inputs["input_ids"].shape[1]:]
    resp = tokenizer.decode(gen, skip_special_tokens=True).strip()

    # extract decision tag
    decision = "UNKNOWN"
    if "<decision>" in resp and "</decision>" in resp:
        start    = resp.find("<decision>") + len("<decision>")
        end      = resp.find("</decision>")
        decision = resp[start:end].strip()

    passed   = decision == case["expect"]
    correct += int(passed)

    print(f"\n  {'─'*54}")
    print(f"  Test {i+1}  : {case['label']}")
    print(f"  Expected : {case['expect']}")
    print(f"  Decision : {decision}  |  {'✅ PASS' if passed else '❌ FAIL'}")
    print(f"  Response :")
    for line in resp.split("\n"):
        print(f"    {line}")

print(f"\n  {'─'*54}")
print(f"  Result: {correct} / {len(test_cases)} passed")


# ── 15. Final Memory + Summary ───────────────────────────────

gc.collect()
torch.cuda.empty_cache()
alloc_end  = torch.cuda.memory_allocated(0) / 1e9
reserv_end = torch.cuda.memory_reserved(0) / 1e9

print("\n" + "=" * 60)
print("  FINETUNING COMPLETE")
print("=" * 60)
print(f"  Best eval loss  : {planner_callback.best_eval:.4f} "
      f"(epoch {planner_callback.best_epoch:.0f})")
print(f"  Final train loss: {m.get('train_loss',0):.4f}")
print(f"  Runtime         : {m.get('train_runtime',0)/60:.1f} min")
print(f"  Adapter (local) : {ADAPTER_DIR}/")
print(f"  Adapter (Drive) : {final_drive_dir}/")
print(f"  Epoch saves     : {GDRIVE_PATH}/epoch_*/")
print(f"  Inference test  : {correct}/{len(test_cases)} passed")
print(f"  GPU memory      : {alloc_end:.2f}GB alloc | "
      f"{reserv_end:.2f}GB reserved")
print("=" * 60)

In [ ]:
# ============================================================
# PUSH ADAPTER TO HF HUB
# ============================================================

import os, shutil
from google.colab import userdata, drive
from huggingface_hub import HfApi, login

drive.mount("/content/drive", force_remount=False)

# ── get token from Colab secrets ─────────────────────────────
HF_TOKEN   = userdata.get("HF_TOKEN")   # set in Colab → Secrets
HF_REPO_ID = "Moodlerz/mistral-planner-aaqa"
GDRIVE_PATH = "/content/drive/MyDrive/planner_finetune"
ADAPTER_DIR = "/content/mistral_planner_adapter"

login(token=HF_TOKEN)
print(f"  ✅ Logged in as Moodlerz")

# ── copy adapter from Drive if not local ─────────────────────
if not os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"):
    shutil.copytree(
        f"{GDRIVE_PATH}/final_adapter",
        ADAPTER_DIR, dirs_exist_ok=True
    )
    print(f"  Copied adapter ← Drive")

# ── create repo ───────────────────────────────────────────────
api = HfApi()
api.create_repo(
    repo_id  = HF_REPO_ID,
    token    = HF_TOKEN,
    exist_ok = True,
    private  = False,
)
print(f"  Repo ready: https://huggingface.co/{HF_REPO_ID}")

# ── write model card ─────────────────────────────────────────
model_card = """---
base_model: mistralai/Mistral-7B-Instruct-v0.3
tags:
- peft
- lora
- question-answering
- decision-planner
- knowledge-graph
license: apache-2.0
---

# Mistral-7B — KG-Grounded Decision Planner

LoRA adapter fine-tuned on Mistral-7B-Instruct-v0.3 for
knowledge-graph-grounded decision planning.

## Task
Given a user query and KG triples retrieved from a knowledge graph,
the model decides:
- **ANSWER** — graph has sufficient evidence
- **ASK** — graph is partial, clarification needed
- **ABSTAIN** — topic absent from graph entirely

## Training
- 37,264 samples across QuAC, SHaRC, HotpotQA, ContractNLI
- Single-turn and multi-turn conversational settings
- KG triples as structured context

## Usage
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

MODEL_ID   = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_ID = "Moodlerz/mistral-planner-aaqa"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
base      = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
model = PeftModel.from_pretrained(base, ADAPTER_ID)
model.eval()
```
"""

with open(f"{ADAPTER_DIR}/README.md", "w") as f:
    f.write(model_card)

# ── upload ────────────────────────────────────────────────────
api.upload_folder(
    folder_path = ADAPTER_DIR,
    repo_id     = HF_REPO_ID,
    token       = HF_TOKEN,
)
print(f"  ✅ Pushed → https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# ============================================================
# RELOAD — no quantization, plain bf16
# ============================================================

import os, json, torch, warnings, shutil
warnings.filterwarnings("ignore")

from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

GDRIVE_PATH  = "/content/drive/MyDrive/planner_finetune"
ADAPTER_DIR  = "/content/mistral_planner_adapter"
FT_OUT_DIR   = "/content"
HF_REPO_ID   = "Moodlerz/mistral-planner-aaqa"
MAX_SEQ_LEN  = 1024
USE_BF16     = torch.cuda.is_bf16_supported()




# copy adapter from Drive
if not os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"):
    shutil.copytree(
        f"{GDRIVE_PATH}/final_adapter",
        ADAPTER_DIR, dirs_exist_ok=True
    )
    print("  Copied adapter ← Drive")

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID  = "mistralai/Mistral-7B-Instruct-v0.3"
print(f"  Loading {MODEL_ID} + adapter (plain bf16, no quant) …")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id


# Add this before model loading
from transformers import StoppingCriteria, StoppingCriteriaList

class StopAfterDecision(StoppingCriteria):
    def __init__(self, tokenizer):
        self.stop_ids = tokenizer.encode(
            "</justification>", add_special_tokens=False
        )

    def __call__(self, input_ids, scores, **kwargs):
        if len(input_ids[0]) < len(self.stop_ids):
            return False
        return (input_ids[0][-len(self.stop_ids):].tolist()
                == self.stop_ids)

stop_criteria = StoppingCriteriaList([StopAfterDecision(tokenizer)])


base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype  = torch.bfloat16 if USE_BF16 else torch.float16,
    device_map   = "auto",
    trust_remote_code = True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

alloc = torch.cuda.memory_allocated(0) / 1e9
print(f"  ✅ Ready  |  VRAM: {alloc:.1f} GB")

# ── Few-shot system prompt ────────────────────────────────────
SYSTEM_PROMPT = """You are a decision planner for a question-answering system.

Your task: given a user query, search the knowledge graph for relevant nodes, evaluate what information is present and what is missing, then decide the correct action.

Decision logic:
- If the graph contains a complete path connecting known entities to an answer → ANSWER
- If the graph contains the topic but key linking variables are missing → ASK
- If the graph has no relevant nodes or the topic is entirely absent → ABSTAIN

Output format (strictly follow this):
<reasoning>
Step 1 — Query subject: identify what the query is asking about
Step 2 — Graph search: what nodes were found, what connections exist
Step 3 — Variable check: what is known, what is missing
Step 4 — Decision rationale: why this action is correct
</reasoning>
<decision>
ANSWER | ASK | ABSTAIN
</decision>
<justification>
One sentence grounded in the graph evidence.
</justification>

Rules:
- Reasoning must reference actual graph content, not generic statements
- Never say "unspecified variables" — name the specific missing variable
- If graph_context is empty, default to ABSTAIN unless context is clearly partial (then ASK)
- Do not use prior world knowledge — only the graph context provided


"""

print("\n  ✅ System prompt with few-shot examples ready")
print(f"  Prompt length: ~{len(SYSTEM_PROMPT.split())} words")

In [ ]:
# run this to confirm nothing is being truncated
test_msg = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": (
        "<query>\nWho wrote Fear and Loathing in Las Vegas?\n</query>\n\n"
        "<known_variables>\nFear and Loathing in Las Vegas\n</known_variables>\n\n"
        "<graph_context>\nthompson | write | fear and loathing in las vegas\n</graph_context>\n\n"
        "<missing_variables>\nnone\n</missing_variables>\n\n"
        "Search the graph context for relevant nodes and decide the correct action."
    )}
]
formatted = tokenizer.apply_chat_template(
    test_msg, tokenize=False, add_generation_prompt=True
)
token_len = len(tokenizer(formatted)["input_ids"])
print(f"  Full prompt tokens : {token_len}")
print(f"  MAX_SEQ_LEN        : {MAX_SEQ_LEN}")
print(f"  Headroom for gen   : {MAX_SEQ_LEN - token_len} tokens")
print(f"  {'✅ OK' if MAX_SEQ_LEN - token_len >= 400 else '❌ TOO SMALL — increase MAX_SEQ_LEN'}")